<a href="https://colab.research.google.com/github/adminsanjay/ML-projects/blob/main/lstm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ================================
# LSTM AUDIO CLASSIFIER
# ================================

import kagglehub
path = kagglehub.dataset_download("mohammedabdeldayem/the-fake-or-real-dataset")

import os
import librosa
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, LSTM, Input
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from sklearn.metrics import accuracy_score, classification_report

# ================================
# DATA GENERATOR (same)
# ================================
class AudioDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, file_paths, labels, batch_size=32, sr=16000, n_mfcc=40, target_frames=64):
        self.file_paths = file_paths
        self.labels = labels
        self.batch_size = batch_size
        self.sr = sr
        self.n_mfcc = n_mfcc
        self.target_frames = target_frames
        self.indices = np.arange(len(self.file_paths))

    def __len__(self):
        return int(np.ceil(len(self.file_paths) / self.batch_size))

    def __getitem__(self, index):
        batch_indices = self.indices[index * self.batch_size:(index + 1) * self.batch_size]
        batch_paths = [self.file_paths[i] for i in batch_indices]
        batch_labels = [self.labels[i] for i in batch_indices]

        X = np.empty((len(batch_paths), self.n_mfcc, self.target_frames, 1))
        y = np.empty((len(batch_paths), 2))

        for i, path in enumerate(batch_paths):
            try:
                audio, _ = librosa.load(path, sr=self.sr)

                if len(audio) < self.sr:
                    audio = np.pad(audio, (0, self.sr - len(audio)))

                mfcc = librosa.feature.mfcc(y=audio, sr=self.sr, n_mfcc=self.n_mfcc)
                mfcc = (mfcc - np.mean(mfcc)) / (np.std(mfcc) + 1e-8)

                if mfcc.shape[1] < self.target_frames:
                    mfcc = np.pad(mfcc, ((0, 0), (0, self.target_frames - mfcc.shape[1])))
                else:
                    mfcc = mfcc[:, :self.target_frames]

                X[i] = np.expand_dims(mfcc, axis=-1)

            except:
                X[i] = np.zeros((self.n_mfcc, self.target_frames, 1))

            y[i] = tf.keras.utils.to_categorical(batch_labels[i], 2)

        return X, y

# ================================
# PATH LOADER
# ================================
def get_paths_and_labels(base_path):
    paths, labels = [], []

    for label_name in ['real', 'fake']:
        label_dir = os.path.join(base_path, label_name)
        if not os.path.exists(label_dir):
            continue

        label = 0 if label_name == 'real' else 1

        for file in os.listdir(label_dir):
            paths.append(os.path.join(label_dir, file))
            labels.append(label)

    return paths, labels

# ================================
# LSTM MODEL
# ================================
def create_lstm_model(input_shape):
    model = Sequential([
        Input(shape=input_shape),
        tf.keras.layers.Reshape((input_shape[1], input_shape[0])),

        LSTM(128, return_sequences=True),
        Dropout(0.2),

        LSTM(64),
        Dropout(0.2),

        Dense(32, activation='relu'),
        Dense(2, activation='softmax')
    ])

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# ================================
# MAIN
# ================================
dataset_dir = "/kaggle/input/the-fake-or-real-dataset/for-rerec/for-rerecorded"

train_paths, train_labels = get_paths_and_labels(os.path.join(dataset_dir, 'training'))
val_paths, val_labels = get_paths_and_labels(os.path.join(dataset_dir, 'validation'))
test_paths, test_labels = get_paths_and_labels(os.path.join(dataset_dir, 'testing'))

train_gen = AudioDataGenerator(train_paths, train_labels)
val_gen = AudioDataGenerator(val_paths, val_labels)
test_gen = AudioDataGenerator(test_paths, test_labels)

model = create_lstm_model((40,64,1))

checkpoint = ModelCheckpoint("lstm_model.keras", save_best_only=True, monitor='val_accuracy')
early_stop = EarlyStopping(patience=5, restore_best_weights=True)

model.fit(train_gen, validation_data=val_gen, epochs=20, callbacks=[checkpoint, early_stop])

# Evaluation
pred_probs = model.predict(test_gen)
pred = np.argmax(pred_probs, axis=1)

print("Accuracy:", accuracy_score(test_labels, pred))
print(classification_report(test_labels, pred))

Using Colab cache for faster access to the 'the-fake-or-real-dataset' dataset.


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/20
319/319 ━━━━━━━━━━━━━━━━━━━━ 0s 583ms/step - accuracy: 0.4754 - loss: 0.7015

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


319/319 ━━━━━━━━━━━━━━━━━━━━ 237s 721ms/step - accuracy: 0.4975 - loss: 0.6981 - val_accuracy: 0.5094 - val_loss: 0.7323
Epoch 2/20
319/319 ━━━━━━━━━━━━━━━━━━━━ 126s 395ms/step - accuracy: 0.5407 - loss: 0.6977 - val_accuracy: 0.4929 - val_loss: 0.6972
Epoch 3/20
319/319 ━━━━━━━━━━━━━━━━━━━━ 138s 383ms/step - accuracy: 0.5138 - loss: 0.6963 - val_accuracy: 0.5254 - val_loss: 0.6919
Epoch 4/20
319/319 ━━━━━━━━━━━━━━━━━━━━ 138s 372ms/step - accuracy: 0.5460 - loss: 0.6997 - val_accuracy: 0.4906 - val_loss: 0.6952
Epoch 5/20
319/319 ━━━━━━━━━━━━━━━━━━━━ 144s 378ms/step - accuracy: 0.5426 - loss: 0.6935 - val_accuracy: 0.7086 - val_loss: 0.6459
Epoch 6/20
319/319 ━━━━━━━━━━━━━━━━━━━━ 123s 386ms/step - accuracy: 0.6600 - loss: 0.6308 - val_accuracy: 0.7242 - val_loss: 0.5595
Epoch 7/20
319/319 ━━━━━━━━━━━━━━━━━━━━ 158s 439ms/step - accuracy: 0.7086 - loss: 0.5774 - val_accuracy: 0.7028 - val_loss: 0.5891
Epoch 8/20
319/319 ━━━━━━━━━━━━━━━━━━━━ 146s 458ms/step - accuracy: 0.7784 - loss: 0.48

In [ ]:
import os
import numpy as np
import tensorflow as tf
import librosa
import soundfile as sf
def convert_to_wav(input_path, output_path="temp_converted.wav"):
    audio, sr = librosa.load(input_path, sr=16000)
    sf.write(output_path, audio, sr)
    return output_path
def predict_audio(audio_path, vgg_path='/content/vgg16_model.keras', lstm_path='/content/lstm_model.keras'):
    wav_path = convert_to_wav(audio_path)
    vgg_model = tf.keras.models.load_model(vgg_path)
    lstm_model = tf.keras.models.load_model(lstm_path)
    y, sr = librosa.load(wav_path, sr=16000)
    if len(y) < sr:
        y = np.pad(y, (0, sr - len(y)))
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
    mfcc = (mfcc - np.mean(mfcc)) / (np.std(mfcc) + 1e-8)
    if mfcc.shape[1] < 64:
        mfcc = np.pad(mfcc, ((0, 0), (0, 64 - mfcc.shape[1])))
    else:
        mfcc = mfcc[:, :64]
    spectrogram = np.expand_dims(np.array([mfcc]), axis=-1)
    vgg_pred = vgg_model.predict(spectrogram, verbose=0)[0]
    lstm_pred = lstm_model.predict(spectrogram, verbose=0)[0]
    vgg_class = np.argmax(vgg_pred)
    lstm_class = np.argmax(lstm_pred)
    vgg_res = "FAKE" if vgg_class == 1 else "REAL"
    lstm_res = "FAKE" if lstm_class == 1  else "REAL"
    vgg_conf = vgg_pred[vgg_class] * 100
    lstm_conf = lstm_pred[lstm_class] * 100
    print(f"File: {audio_path}")
    print(f"VGG16 Prediction: {vgg_res} ({vgg_conf:.2f}%)")
    print(f"LSTM Prediction: {lstm_res} ({lstm_conf:.2f}%)")
    if vgg_class == lstm_class:
        print(f"Consensus: STRONG {vgg_res}")
    else:
        print("Consensus: INCONCLUSIVE")
    if os.path.exists(wav_path):
        os.remove(wav_path)
if __name__ == "__main__":
    input_file = input("Enter the path to the audio file: ")
    predict_audio(input_file)

Enter the path to the audio file: /content/sanjay.mp3


File: /content/sanjay.mp3
VGG16 Prediction: REAL (91.37%)
LSTM Prediction: REAL (84.68%)
Consensus: STRONG REAL
